In [1]:
import pandas as pd
import numpy as np
import glob
import os
from tqdm import tqdm

In [2]:
!pwd

/storage/Arushi/090526_EvoAge/kg_formation/processed_data_relation_wise_merge/generalised


In [20]:
BASE_DIR     = '/storage/Arushi/090526_EvoAge/kg_formation/'
MAPPING_DIR  = BASE_DIR + 'data_collection/databases_for_mapping/'
PROC_DIR     = BASE_DIR + 'processed_data/'

# ── Output path ───────────────────────────────────────────────────────────────
OUT_PATH = BASE_DIR + 'processed_data_relation_wise_merge/generalised/MIRNA_GENE/Human_mirna_gene.csv'

# ── Required output schema ────────────────────────────────────────────────────
REQUIRED_COLS = [
    'head', 'relation', 'tail',
    'head_type', 'relation_type', 'tail_type',
    'kg_source', 
    'head_id_is', 'tail_id_is',
    'head_detail_name', 'tail_detail_name', 'kg_type','species'
]

# mirna

In [21]:
mirna = pd.read_csv(PROC_DIR + 'mirTARbase/hsap/miRTarBase_Mirna_Gene.csv')

mirna.columns = mirna.columns.str.lower()
mirna = mirna.rename(columns={
    'headtype': 'head_type',
    'tailtype': 'tail_type',
})
mirna = mirna[~mirna['head_detail_name'].isna()]
mirna['kg_type'] = 'Generalised'
mirna['head_type'] = 'Mirna'
mirna['species'] = 'Homo sapiens'
mirna['tail_id_is'] = 'NCBI_ID'
mirna['head_id_is'] = 'miRTarBase_ID'
mirna['kg_source'] = 'miRTarBase'

print(f"mirna: {len(mirna):,} rows")
mirna

mirna: 3,974,680 rows


,head,relation,tail,head_type,tail_type,kg_source,head_detail_name,tail_detail_name,species,kg_type,tail_id_is,head_id_is
0,hsa-miR-122-5p,Mirna_Gene,SLC7A1,Mirna,Gene,miRTarBase,hsa-miR-122-5p,solute carrier family 7 member 1,Homo sapiens,Generalised,NCBI_ID,miRTarBase_ID
1,hsa-miR-122-5p,Mirna_Gene,SLC7A1,Mirna,Gene,miRTarBase,hsa-miR-122-5p,solute carrier family 7 member 1,Homo sapiens,Generalised,NCBI_ID,miRTarBase_ID
2,hsa-miR-122-5p,Mirna_Gene,SLC7A1,Mirna,Gene,miRTarBase,hsa-miR-122-5p,solute carrier family 7 member 1,Homo sapiens,Generalised,NCBI_ID,miRTarBase_ID
3,hsa-miR-122-5p,Mirna_Gene,SLC7A1,Mirna,Gene,miRTarBase,hsa-miR-122-5p,solute carrier family 7 member 1,Homo sapiens,Generalised,NCBI_ID,miRTarBase_ID
4,hsa-miR-122-5p,Mirna_Gene,SLC7A1,Mirna,Gene,miRTarBase,hsa-miR-122-5p,solute carrier family 7 member 1,Homo sapiens,Generalised,NCBI_ID,miRTarBase_ID
...,...,...,...,...,...,...,...,...,...,...,...,...
3974675,hsa-miR-549,Mirna_Gene,ZZZ3,Mirna,Gene,miRTarBase,hsa-miR-549,zinc finger ZZ-type containing 3,Homo sapiens,Generalised,NCBI_ID,miRTarBase_ID
3974676,hsa-miR-556-5p,Mirna_Gene,ZZZ3,Mirna,Gene,miRTarBase,hsa-miR-556-5p,zinc finger ZZ-type containing 3,Homo sapiens,Generalised,NCBI_ID,miRTarBase_ID
3974677,hsa-miR-558,Mirna_Gene,ZZZ3,Mirna,Gene,miRTarBase,hsa-miR-558,zinc finger ZZ-type containing 3,Homo sapiens,Generalised,NCBI_ID,miRTarBase_ID
3974678,hsa-miR-590-3p,Mirna_Gene,ZZZ3,Mirna,Gene,miRTarBase,hsa-miR-590-3p,zinc finger ZZ-type containing 3,Homo sapiens,Generalised,NCBI_ID,miRTarBase_ID


In [22]:
SOURCE_DFS = [
    ('mirna',     mirna),
]

for name, df in SOURCE_DFS:
    dupes = df.columns[df.columns.duplicated(keep=False)].unique().tolist()
    if dupes:
        print(f"\n[{name}] duplicate columns: {dupes}")
        for col in dupes:
            for i, (_, s) in enumerate(df.loc[:, df.columns == col].items()):
                print(f"  '{col}' occurrence {i} → sample: {s.dropna().head(3).tolist()}")
    else:
        print(f"[{name}] ✓ no duplicates")

[mirna] ✓ no duplicates


# Consolidate into Unified Schem

In [23]:
# List all source DataFrames to include
source_dfs = [
    mirna,
]

aligned = []
for df in source_dfs:
    df = df.copy()

    for col in REQUIRED_COLS:
        if col not in df.columns:
            df[col] = None       
    aligned.append(df[REQUIRED_COLS])

final_df = pd.concat(aligned, ignore_index=True)
print(f"Consolidated rows: {len(final_df):,}")
final_df

Consolidated rows: 3,974,680


,head,relation,tail,head_type,relation_type,tail_type,kg_source,head_id_is,tail_id_is,head_detail_name,tail_detail_name,kg_type,species
0,hsa-miR-122-5p,Mirna_Gene,SLC7A1,Mirna,None,Gene,miRTarBase,miRTarBase_ID,NCBI_ID,hsa-miR-122-5p,solute carrier family 7 member 1,Generalised,Homo sapiens
1,hsa-miR-122-5p,Mirna_Gene,SLC7A1,Mirna,None,Gene,miRTarBase,miRTarBase_ID,NCBI_ID,hsa-miR-122-5p,solute carrier family 7 member 1,Generalised,Homo sapiens
2,hsa-miR-122-5p,Mirna_Gene,SLC7A1,Mirna,None,Gene,miRTarBase,miRTarBase_ID,NCBI_ID,hsa-miR-122-5p,solute carrier family 7 member 1,Generalised,Homo sapiens
3,hsa-miR-122-5p,Mirna_Gene,SLC7A1,Mirna,None,Gene,miRTarBase,miRTarBase_ID,NCBI_ID,hsa-miR-122-5p,solute carrier family 7 member 1,Generalised,Homo sapiens
4,hsa-miR-122-5p,Mirna_Gene,SLC7A1,Mirna,None,Gene,miRTarBase,miRTarBase_ID,NCBI_ID,hsa-miR-122-5p,solute carrier family 7 member 1,Generalised,Homo sapiens
...,...,...,...,...,...,...,...,...,...,...,...,...,...
3974675,hsa-miR-549,Mirna_Gene,ZZZ3,Mirna,None,Gene,miRTarBase,miRTarBase_ID,NCBI_ID,hsa-miR-549,zinc finger ZZ-type containing 3,Generalised,Homo sapiens
3974676,hsa-miR-556-5p,Mirna_Gene,ZZZ3,Mirna,None,Gene,miRTarBase,miRTarBase_ID,NCBI_ID,hsa-miR-556-5p,zinc finger ZZ-type containing 3,Generalised,Homo sapiens
3974677,hsa-miR-558,Mirna_Gene,ZZZ3,Mirna,None,Gene,miRTarBase,miRTarBase_ID,NCBI_ID,hsa-miR-558,zinc finger ZZ-type containing 3,Generalised,Homo sapiens
3974678,hsa-miR-590-3p,Mirna_Gene,ZZZ3,Mirna,None,Gene,miRTarBase,miRTarBase_ID,NCBI_ID,hsa-miR-590-3p,zinc finger ZZ-type containing 3,Generalised,Homo sapiens


# Sanity Check — Distinct Values

In [24]:
for col in ['relation', 'head_type', 'tail_type', 'relation_type', 'kg_source', 'head_id_is', 'tail_id_is']:
    print(f"{col:20s}: {set(final_df[col])}")

relation            : {'Mirna_Gene'}
head_type           : {'Mirna'}
tail_type           : {'Gene'}
relation_type       : {None}
kg_source           : {'miRTarBase'}
head_id_is          : {'miRTarBase_ID'}
tail_id_is          : {'NCBI_ID'}


In [25]:
# Step 4: drop unresolvable rows
before = len(final_df)
final_df = final_df[~final_df['tail_detail_name'].isna()].reset_index(drop=True)
print(f"Dropped {before - len(final_df):,} unresolvable rows → {len(final_df):,} remaining")

Dropped 0 unresolvable rows → 3,974,680 remaining


# NaN Audit (pre-dedup)

In [26]:
true_nan   = final_df.isna().sum()
string_nan = final_df.apply(lambda col: col.astype(str).str.upper().eq('NAN').sum())

pd.DataFrame({
    'NaN_count':          true_nan,
    "'NAN'_string_count": string_nan,
    'Total_NaN_like':     true_nan + string_nan
})

,NaN_count,'NAN'_string_count,Total_NaN_like
head,0,0,0
relation,0,0,0
tail,0,0,0
head_type,0,0,0
relation_type,3974680,0,3974680
tail_type,0,0,0
kg_source,0,0,0
head_id_is,0,0,0
tail_id_is,0,0,0
head_detail_name,0,0,0


In [27]:
SOURCE_DFS = [
    ('final_df',     final_df),
]

for name, df in SOURCE_DFS:
    dupes = df.columns[df.columns.duplicated(keep=False)].unique().tolist()
    if dupes:
        print(f"\n[{name}] duplicate columns: {dupes}")
        for col in dupes:
            for i, (_, s) in enumerate(df.loc[:, df.columns == col].items()):
                print(f"  '{col}' occurrence {i} → sample: {s.dropna().head(3).tolist()}")
    else:
        print(f"[{name}] ✓ no duplicates")

[final_df] ✓ no duplicates


# Deduplication

In [28]:
def merge_sources(x):
    """Combine unique, non-null source labels with '::' delimiter."""
    return '::'.join(sorted(set(x.dropna())))

group_cols = ['head', 'relation', 'tail']

final_df_dedup = final_df.groupby(group_cols, as_index=False).agg({
    'head_type':        'first',
    'relation_type':    'first',
    'tail_type':        'first',
    'kg_source':        merge_sources,
    'kg_type':          merge_sources,   # ← changed from 'first'
    'head_id_is':       'first',
    'tail_id_is':       'first',
    'head_detail_name': 'first',
    'tail_detail_name': 'first',
    'species': 'first'
})

print(f"Before dedup: {len(final_df):,}  |  After dedup: {len(final_df_dedup):,}")
final_df_dedup.head(3)

Before dedup: 3,974,680  |  After dedup: 1,717,651


,head,relation,tail,head_type,relation_type,tail_type,kg_source,kg_type,head_id_is,tail_id_is,head_detail_name,tail_detail_name,species
0,hsa-let-7a,Mirna_Gene,A1CF,Mirna,None,Gene,miRTarBase,Generalised,miRTarBase_ID,NCBI_ID,hsa-let-7a,APOBEC1 complementation factor,Homo sapiens
1,hsa-let-7a,Mirna_Gene,AAK1,Mirna,None,Gene,miRTarBase,Generalised,miRTarBase_ID,NCBI_ID,hsa-let-7a,AP2 associated kinase 1,Homo sapiens
2,hsa-let-7a,Mirna_Gene,AASS,Mirna,None,Gene,miRTarBase,Generalised,miRTarBase_ID,NCBI_ID,hsa-let-7a,aminoadipate-semialdehyde synthase,Homo sapiens


In [29]:
true_nan   = final_df_dedup.isna().sum()
string_nan = final_df_dedup.apply(lambda col: col.astype(str).str.upper().eq('NAN').sum())

pd.DataFrame({
    'NaN_count':          true_nan,
    "'NAN'_string_count": string_nan,
    'Total_NaN_like':     true_nan + string_nan
})

,NaN_count,'NAN'_string_count,Total_NaN_like
head,0,0,0
relation,0,0,0
tail,0,0,0
head_type,0,0,0
relation_type,1717651,0,1717651
tail_type,0,0,0
kg_source,0,0,0
kg_type,0,0,0
head_id_is,0,0,0
tail_id_is,0,0,0


In [30]:
print("kg_source values present:", set(final_df_dedup['kg_source']), final_df_dedup['kg_source'].value_counts())

kg_source values present: {'miRTarBase'} kg_source
miRTarBase    1717651
Name: count, dtype: int64


In [31]:
print("kg_source values present:", set(final_df_dedup['kg_type']), final_df_dedup['kg_type'].value_counts())

kg_source values present: {'Generalised'} kg_type
Generalised    1717651
Name: count, dtype: int64


In [32]:
final_df_dedup.to_csv(OUT_PATH, index=False)
print(f"Saved {len(final_df_dedup):,} rows → {OUT_PATH}")

Saved 1,717,651 rows → /storage/Arushi/090526_EvoAge/kg_formation/processed_data_relation_wise_merge/generalised/MIRNA_GENE/Human_mirna_gene.csv


In [33]:
OUT_PATH

'/storage/Arushi/090526_EvoAge/kg_formation/processed_data_relation_wise_merge/generalised/MIRNA_GENE/Human_mirna_gene.csv'